# B2 – Modello lineare (loss L1, penalità L2)

*Progetto: tokenizzazione dei network flow CERN, Gruppo 1. Baseline 2 di 5.*

**Idea.** Prevedere `delta_rtt` come **somma di effetti**: un'intercetta, un termine per `start_rtt` e un coefficiente per ogni valore di ogni campo categorico (sito, rete, applicazione, hostname, porte…). Nessuna interazione tra campi: l'effetto del server non dipende dal client. Con la loss L1 il modello stima la **mediana condizionata** come funzione lineare (regressione mediana).

**Dati.** I file codificati prodotti da `prepare.py` (stesse righe e stessi codici per B2, B3 e B4):
`train_enc` (~3 M flow, primo 90 % del train) per imparare, `tune_enc` (~1 M flow, ultimo 10 % del train) per le scelte e l'early stopping, `val_enc` (tutta la validation, 10,07 M flow) per il risultato, confrontabile con B0 (MAE 3,298) e B1 (MAE 2,209).

**Struttura.** §0 Dati · §1 Caricamento · §2 `start_rtt` · §3 Modello · §4 Addestramento · §5 Scelta di λ · §6 Modello finale · §7 Valutazione sulla validation · §8 Lettura del modello.

## 0. Collegamento ai dati codificati

Apro una connessione DuckDB e creo una **vista** per ciascuno dei tre file prodotti da `prepare.py` (`train_enc`, `tune_enc`, `val_enc`). Una vista è solo un nome che punta al file: i dati restano sul disco e DuckDB legge solo ciò che le query chiedono. Non riparto da `flows_prepared.parquet`: rifare la codifica qui rischierebbe vocabolari o campioni diversi da quelli di B3 e B4.

Controllo: righe attese 3.000.316 / 1.000.351 / 10.069.254; 22 colonne (`riga`, `flow_id`, `t_start`, target, `start_rtt`, 13 codici `id_*`, 4 colonne in chiaro solo per i sottogruppi).

In [1]:
from pathlib import Path
import duckdb

PROJECT_ROOT = Path.cwd().parent                     # il notebook sta in results/
ENC_DIR = PROJECT_ROOT / "data" / "encoded"

con = duckdb.connect()
con.execute("SET threads TO 8")
con.execute("SET memory_limit = '8GB'")

for nome in ["train_enc", "tune_enc", "val_enc"]:
    file = (ENC_DIR / f"{nome}.parquet").as_posix()
    con.execute(f"CREATE OR REPLACE VIEW {nome} AS SELECT * FROM read_parquet('{file}')")

display(con.execute("""
    SELECT 'train_enc' AS insieme, COUNT(*) AS n FROM train_enc
    UNION ALL SELECT 'tune_enc', COUNT(*) FROM tune_enc
    UNION ALL SELECT 'val_enc',  COUNT(*) FROM val_enc
""").df())

display(con.execute("DESCRIBE val_enc").df()[["column_name", "column_type"]])

,insieme,n
0,train_enc,3000316
1,tune_enc,1000351
2,val_enc,10069254


,column_name,column_type
0,riga,BIGINT
1,flow_id,VARCHAR
2,t_start,TIMESTAMP WITH TIME ZONE
3,delta_rtt,DOUBLE
4,start_rtt,DOUBLE
5,id_ip_version,INTEGER
6,id_experiment,INTEGER
7,id_exp_act,INTEGER
8,id_application,INTEGER
9,id_hostname,INTEGER


## 1. Caricamento in memoria

Leggo da `meta.json` l'elenco dei 13 input categorici (`id_*`) e carico, per ogni insieme, solo target (`delta_rtt`), `start_rtt` e codici, come array numpy (`float32` per i numeri, `int32` per i codici, per dimezzare la memoria).

Le colonne sono elencate **esplicitamente**: `an_durata_s` (durata, nota solo a fine flow) non deve mai entrare negli input (leakage). Per la validation tengo anche `riga`, la chiave per riattaccare le previsioni ai flow e calcolare le metriche per sottogruppo.

In [2]:
import json
import numpy as np

meta = json.loads((ENC_DIR / "meta.json").read_text(encoding="utf-8"))
COL_ID = meta["colonne_id"]
print(len(COL_ID), "input categorici:", COL_ID)

def carica(nome):
    df = con.execute(f"SELECT riga, delta_rtt, start_rtt, {', '.join(COL_ID)} FROM {nome}").df()
    riga = df["riga"].to_numpy()
    y    = df["delta_rtt"].to_numpy(np.float32)
    rtt  = df["start_rtt"].to_numpy(np.float32)
    X    = df[COL_ID].to_numpy(np.int32)
    return riga, y, rtt, X

_,       y_tr, rtt_tr, X_tr = carica("train_enc")
_,       y_tu, rtt_tu, X_tu = carica("tune_enc")
riga_va, y_va, rtt_va, X_va = carica("val_enc")

print("forme di X:", X_tr.shape, X_tu.shape, X_va.shape)
print(f"memoria X_va: {X_va.nbytes / 1e9:.2f} GB")
print("mediana di y_tr:", np.median(y_tr))

13 input categorici: ['id_ip_version', 'id_experiment', 'id_exp_act', 'id_application', 'id_hostname', 'id_server_site', 'id_client_site', 'id_server_L2', 'id_client_L2', 'id_porta_servizio', 'id_porta_client', 'id_verso', 'id_verso_incerto']
forme di X: (3000316, 13) (1000351, 13) (10069254, 13)
memoria X_va: 0.52 GB
mediana di y_tr: 4.405


## 2. Trasformazione di `start_rtt`

Nel modello lineare `start_rtt` entra con un solo coefficiente $w$. Uso $\log(\text{start\_rtt})$ perché conti il **rapporto** tra RTT e non la differenza in ms (1 → 10 ms pesa come 10 → 100 ms), e per comprimere la coda lunga. Poi standardizzo:

$$x_{\text{rtt}} = \frac{\log(\text{start\_rtt}) - \mu}{\sigma}$$

con $\mu$ e $\sigma$ calcolate **solo su `train_enc`** e applicate identiche a tune e validation. La standardizzazione aiuta l'ottimizzazione e rende la penalità L2 indipendente dall'unità di misura. Prima controllo che non ci siano RTT nulli (il log di 0 non esiste).

In [3]:
print("start_rtt minimo:", rtt_tr.min(), rtt_tu.min(), rtt_va.min())
assert min(rtt_tr.min(), rtt_tu.min(), rtt_va.min()) > 0, "start_rtt nullo: il log non è definito"

log_tr = np.log(rtt_tr)
mu, sigma = float(log_tr.mean()), float(log_tr.std())

def trasforma_rtt(rtt):
    return ((np.log(rtt) - mu) / sigma).astype(np.float32)

x_tr = trasforma_rtt(rtt_tr)
x_tu = trasforma_rtt(rtt_tu)
x_va = trasforma_rtt(rtt_va)

print(f"mu = {mu:.3f} (RTT tipico ≈ {np.exp(mu):.1f} ms) | sigma = {sigma:.3f}")
for nome, x in [("train", x_tr), ("tune", x_tu), ("val", x_va)]:
    print(f"{nome:5s} media = {x.mean():+.3f} | std = {x.std():.3f}")

start_rtt minimo: 0.007 0.007 0.008
mu = 2.823 (RTT tipico ≈ 16.8 ms) | sigma = 1.653
train media = +0.000 | std = 1.000
tune  media = -0.094 | std = 0.970
val   media = -0.010 | std = 0.907


## 3. Il modello

$$\hat y = b + w\, x_{\text{rtt}} + \sum_{j=1}^{13} \theta_j[\text{codice}_j]$$

Intercetta $b$, coefficiente $w$ di `start_rtt` e un coefficiente per ogni valore di ogni campo: una tabella `nn.Embedding(n, 1)` per colonna (equivalente al one-hot seguito da un prodotto scalare, senza costruire la matrice di zeri). Le tabelle sono **separate anche per server e client**: il vocabolario condiviso dà lo stesso codice a un sito nei due ruoli, ma il suo effetto su `delta_rtt` può essere diverso.

Le dimensioni delle tabelle vengono da `meta.json` (4 codici riservati + valori frequenti). I coefficienti partono da 0 e l'intercetta dalla mediana del train: il modello non addestrato coincide con B0, e lo verifico.

In [4]:
import torch
import torch.nn as nn

vocab = json.loads((ENC_DIR / "vocabolari.json").read_text(encoding="utf-8"))

# colonna -> vocabolario (es. 'server_site' -> 'site') -> numero di codici
col_vocab = {c: nome for nome, colonne in vocab["colonne"].items() for c in colonne}
N_CODICI  = [meta["n_codici"][col_vocab[c.removeprefix("id_")]] for c in COL_ID]

class Lineare(nn.Module):
    def __init__(self, n_codici, intercetta_iniziale):
        super().__init__()
        self.emb = nn.ModuleList([nn.Embedding(n, 1) for n in n_codici])
        for e in self.emb:
            nn.init.zeros_(e.weight)
        self.w = nn.Parameter(torch.zeros(1))
        self.b = nn.Parameter(torch.tensor([intercetta_iniziale]))

    def forward(self, x_cat, x_rtt):
        out = self.b + self.w * x_rtt
        for j, e in enumerate(self.emb):
            out = out + e(x_cat[:, j]).squeeze(1)
        return out

MEDIANA_TR = float(np.median(y_tr))
modello = Lineare(N_CODICI, MEDIANA_TR)

print(dict(zip(COL_ID, N_CODICI)))
print("parametri:", sum(p.numel() for p in modello.parameters()))

# controllo: il modello non addestrato deve prevedere la mediana per ogni flow
with torch.no_grad():
    prova = modello(torch.from_numpy(X_tr[:5]).long(), torch.from_numpy(x_tr[:5]))
print("previsioni iniziali:", prova.numpy(), "| mediana:", MEDIANA_TR)

{'id_ip_version': 6, 'id_experiment': 10, 'id_exp_act': 33, 'id_application': 60, 'id_hostname': 690, 'id_server_site': 87, 'id_client_site': 87, 'id_server_L2': 344, 'id_client_L2': 344, 'id_porta_servizio': 50, 'id_porta_client': 8, 'id_verso': 6, 'id_verso_incerto': 6}
parametri: 1733
previsioni iniziali: [4.405 4.405 4.405 4.405 4.405] | mediana: 4.40500020980835


/var/folders/ws/68j119dn1_549x290g8q68d40000gn/T/ipykernel_39810/687143294.py:33: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:214.)
  prova = modello(torch.from_numpy(X_tr[:5]).long(), torch.from_numpy(x_tr[:5]))


## 4. Addestramento

`prevedi` calcola le previsioni a blocchi, senza gradienti. `addestra(lam)` crea un modello nuovo e lo addestra con **Adam** su mini-batch di 4.096 flow di `train_enc` in ordine casuale, con **loss L1** (MAE). La penalità L2 di intensità `lam` (`weight_decay`) è applicata solo ai coefficienti delle tabelle, non a intercetta e $w$.

A fine epoca misura la MAE su `tune_enc`, salva i coefficienti migliori e si ferma dopo `pazienza` epoche senza miglioramenti (**early stopping**).

### 4.0 Prima prova: passo fisso (λ = 0)

Verifica che l'addestramento funzioni. Riferimento: B0 sul tune.

*Esito:* la MAE scende da 3,209 (B0) a ~1,764 già alla prima epoca, poi oscilla di qualche millesimo (1,764–1,768): il modello (1.733 parametri) converge subito; le oscillazioni dipendono dalla loss L1, che fa passi di grandezza ~`lr` qualunque sia l'errore. Non è overfitting.

In [5]:
import copy
import time

# train in tensori PyTorch una volta sola (gli indici di nn.Embedding devono essere int64)
Xt_tr = torch.from_numpy(X_tr).long()
xt_tr = torch.from_numpy(x_tr)
yt_tr = torch.from_numpy(y_tr)

@torch.no_grad()
def prevedi(modello, X, x, blocco=500_000):
    """Previsioni a blocchi, per non creare tensori enormi (la validation ha 10 M righe)."""
    modello.eval()
    out = [modello(torch.from_numpy(X[i:i + blocco]).long(), torch.from_numpy(x[i:i + blocco]))
           for i in range(0, len(X), blocco)]
    return torch.cat(out).numpy()

def addestra(lam, lr=0.01, batch=4096, max_epoche=30, pazienza=3, seme=0):
    torch.manual_seed(seme)
    modello = Lineare(N_CODICI, MEDIANA_TR)
    opt = torch.optim.Adam([
        {"params": modello.emb.parameters(),   "weight_decay": lam},   # coefficienti: penalizzati
        {"params": [modello.w, modello.b],     "weight_decay": 0.0},   # w e intercetta: no
    ], lr=lr)

    n = len(yt_tr)
    mae_migliore, epoca_migliore, stato_migliore, storia = float("inf"), 0, None, []

    for epoca in range(1, max_epoche + 1):
        t0 = time.time()
        modello.train()
        ordine = torch.randperm(n)
        for i in range(0, n, batch):
            idx  = ordine[i:i + batch]
            pred = modello(Xt_tr[idx], xt_tr[idx])
            loss = (pred - yt_tr[idx]).abs().mean()
            opt.zero_grad()
            loss.backward()
            opt.step()

        mae_tu = float(np.abs(prevedi(modello, X_tu, x_tu) - y_tu).mean())
        storia.append({"epoca": epoca, "mae_tune": mae_tu, "secondi": time.time() - t0})
        print(f"epoca {epoca:2d} | MAE tune = {mae_tu:.4f} | {time.time() - t0:.0f} s")

        if mae_tu < mae_migliore - 1e-4:
            mae_migliore, epoca_migliore = mae_tu, epoca
            stato_migliore = copy.deepcopy(modello.state_dict())
        elif epoca - epoca_migliore >= pazienza:
            print(f"early stopping: migliore all'epoca {epoca_migliore}")
            break

    modello.load_state_dict(stato_migliore)
    return modello, epoca_migliore, mae_migliore, pd.DataFrame(storia)

import pandas as pd
mae_b0_tune = float(np.abs(MEDIANA_TR - y_tu).mean())
print(f"riferimento B0 sul tune: MAE = {mae_b0_tune:.4f}")

modello, ep, mae_tu, storia = addestra(lam=0.0)

riferimento B0 sul tune: MAE = 3.2094
epoca  1 | MAE tune = 1.7683 | 2 s
epoca  2 | MAE tune = 1.7638 | 2 s
epoca  3 | MAE tune = 1.7678 | 2 s
epoca  4 | MAE tune = 1.7654 | 2 s
epoca  5 | MAE tune = 1.7679 | 2 s
early stopping: migliore all'epoca 2


### 4.1 Funzione aggiornata: passo decrescente

Stessa funzione, con l'opzione `gamma`: a fine epoca il passo `lr` viene moltiplicato per `gamma` (es. 0,5 = dimezzato, `ExponentialLR`). Le ultime epoche fanno correzioni sempre più piccole e i coefficienti si assestano invece di oscillare intorno all'ottimo. Con `gamma = 1` il comportamento è quello di §4.0. Questa definizione sostituisce la precedente.

In [6]:
import copy
import time
import pandas as pd

# train in tensori PyTorch una volta sola (gli indici di nn.Embedding devono essere int64)
Xt_tr = torch.from_numpy(X_tr).long()
xt_tr = torch.from_numpy(x_tr)
yt_tr = torch.from_numpy(y_tr)

@torch.no_grad()
def prevedi(modello, X, x, blocco=500_000):
    """Previsioni a blocchi, per non creare tensori enormi (la validation ha 10 M righe)."""
    modello.eval()
    out = [modello(torch.from_numpy(X[i:i + blocco]).long(), torch.from_numpy(x[i:i + blocco]))
           for i in range(0, len(X), blocco)]
    return torch.cat(out).numpy()

def addestra(lam, lr=0.01, gamma=1.0, batch=4096, max_epoche=30, pazienza=3, seme=0):
    torch.manual_seed(seme)
    modello = Lineare(N_CODICI, MEDIANA_TR)
    opt = torch.optim.Adam([
        {"params": modello.emb.parameters(),   "weight_decay": lam},   # coefficienti: penalizzati
        {"params": [modello.w, modello.b],     "weight_decay": 0.0},   # w e intercetta: no
    ], lr=lr)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=gamma)   # lr × gamma a ogni epoca

    n = len(yt_tr)
    mae_migliore, epoca_migliore, stato_migliore, storia = float("inf"), 0, None, []

    for epoca in range(1, max_epoche + 1):
        t0 = time.time()
        modello.train()
        ordine = torch.randperm(n)
        for i in range(0, n, batch):
            idx  = ordine[i:i + batch]
            pred = modello(Xt_tr[idx], xt_tr[idx])
            loss = (pred - yt_tr[idx]).abs().mean()
            opt.zero_grad()
            loss.backward()
            opt.step()

        mae_tu = float(np.abs(prevedi(modello, X_tu, x_tu) - y_tu).mean())
        lr_epoca = scheduler.get_last_lr()[0]
        storia.append({"epoca": epoca, "lr": lr_epoca, "mae_tune": mae_tu, "secondi": time.time() - t0})
        print(f"epoca {epoca:2d} | lr = {lr_epoca:.5f} | MAE tune = {mae_tu:.4f} | {time.time() - t0:.0f} s")
        scheduler.step()

        if mae_tu < mae_migliore - 1e-4:
            mae_migliore, epoca_migliore = mae_tu, epoca
            stato_migliore = copy.deepcopy(modello.state_dict())
        elif epoca - epoca_migliore >= pazienza:
            print(f"early stopping: migliore all'epoca {epoca_migliore}")
            break

    modello.load_state_dict(stato_migliore)
    return modello, epoca_migliore, mae_migliore, pd.DataFrame(storia)

mae_b0_tune = float(np.abs(MEDIANA_TR - y_tu).mean())
print(f"riferimento B0 sul tune: MAE = {mae_b0_tune:.4f}")

riferimento B0 sul tune: MAE = 3.2094


### 4.2 Prova con passo decrescente (λ = 0, `gamma` = 0,5)

Confronto con la prova a passo fisso (migliore MAE tune 1,7639): ci si aspetta una MAE che scende e si stabilizza invece di oscillare.

*Esito:* MAE tune 1,7631, stabile alla quarta cifra dall'epoca 5. Guadagno minimo (0,0008 ms), ma risultato **stabile**: 1,763 è il limite del modello additivo, non dell'addestramento. Da qui in poi `gamma = 0,5`, al massimo 8 epoche.

In [7]:
modello, ep, mae_tu, storia = addestra(lam=0.0, gamma=0.5, max_epoche=10)
print(f"migliore: epoca {ep}, MAE tune = {mae_tu:.4f}  (passo fisso: 1.7639)")

epoca  1 | lr = 0.01000 | MAE tune = 1.7683 | 2 s
epoca  2 | lr = 0.00500 | MAE tune = 1.7637 | 2 s
epoca  3 | lr = 0.00250 | MAE tune = 1.7641 | 2 s
epoca  4 | lr = 0.00125 | MAE tune = 1.7639 | 2 s
epoca  5 | lr = 0.00063 | MAE tune = 1.7632 | 2 s
epoca  6 | lr = 0.00031 | MAE tune = 1.7632 | 2 s
epoca  7 | lr = 0.00016 | MAE tune = 1.7631 | 2 s
epoca  8 | lr = 0.00008 | MAE tune = 1.7631 | 2 s
epoca  9 | lr = 0.00004 | MAE tune = 1.7632 | 2 s
epoca 10 | lr = 0.00002 | MAE tune = 1.7631 | 2 s
early stopping: migliore all'epoca 7
migliore: epoca 7, MAE tune = 1.7631  (passo fisso: 1.7639)


## 5. Scelta della penalità λ

Addestro il modello (passo dimezzato a ogni epoca, al massimo 8 epoche) con $\lambda \in \{0, 10^{-6}, 10^{-5}, 10^{-4}, 10^{-3}\}$ e confronto le MAE sul tune. La penalità spinge i coefficienti verso 0 con la stessa forza per tutti i valori, mentre i dati li spingono con forza proporzionale a quanti flow li usano: agisce quindi soprattutto sui **valori rari** (*shrinkage*).

**Regola fissata in anticipo:** vince la MAE più bassa; se più valori stanno entro 0,0005 dal migliore (rumore), scelgo il λ più grande (più prudente verso valori rari e drift).

*Esito:* λ ≤ 10⁻⁵ equivalenti (entro 0,0003); 10⁻⁴ peggiora di 0,003, 10⁻³ di 0,019. Scelto **λ = 10⁻⁵**. La penalità non serve in modo apprezzabile: il modello non fa overfitting.

In [8]:
LAMBDA = [0.0, 1e-6, 1e-5, 1e-4, 1e-3]
risultati_lam, modelli_lam = [], {}

for lam in LAMBDA:
    print(f"\n--- lam = {lam:g} ---")
    m, ep, mae, _ = addestra(lam=lam, gamma=0.5, max_epoche=8)
    modelli_lam[lam] = m
    risultati_lam.append({"lam": lam, "epoca_migliore": ep, "mae_tune": mae})

tab_lam = pd.DataFrame(risultati_lam)
tab_lam["distanza_dal_migliore"] = tab_lam["mae_tune"] - tab_lam["mae_tune"].min()

candidati = tab_lam[tab_lam["distanza_dal_migliore"] <= 0.0005]
LAM_SCELTO = float(candidati["lam"].max())

display(tab_lam.round(5))
print("lambda scelto:", LAM_SCELTO)


--- lam = 0 ---
epoca  1 | lr = 0.01000 | MAE tune = 1.7683 | 2 s
epoca  2 | lr = 0.00500 | MAE tune = 1.7637 | 2 s
epoca  3 | lr = 0.00250 | MAE tune = 1.7641 | 2 s
epoca  4 | lr = 0.00125 | MAE tune = 1.7639 | 2 s
epoca  5 | lr = 0.00063 | MAE tune = 1.7632 | 2 s
epoca  6 | lr = 0.00031 | MAE tune = 1.7632 | 2 s
epoca  7 | lr = 0.00016 | MAE tune = 1.7631 | 2 s
epoca  8 | lr = 0.00008 | MAE tune = 1.7631 | 2 s

--- lam = 1e-06 ---
epoca  1 | lr = 0.01000 | MAE tune = 1.7683 | 2 s
epoca  2 | lr = 0.00500 | MAE tune = 1.7637 | 2 s
epoca  3 | lr = 0.00250 | MAE tune = 1.7641 | 2 s
epoca  4 | lr = 0.00125 | MAE tune = 1.7639 | 2 s
epoca  5 | lr = 0.00063 | MAE tune = 1.7631 | 2 s
epoca  6 | lr = 0.00031 | MAE tune = 1.7632 | 2 s
epoca  7 | lr = 0.00016 | MAE tune = 1.7630 | 2 s
epoca  8 | lr = 0.00008 | MAE tune = 1.7631 | 2 s

--- lam = 1e-05 ---
epoca  1 | lr = 0.01000 | MAE tune = 1.7673 | 2 s
epoca  2 | lr = 0.00500 | MAE tune = 1.7634 | 2 s
epoca  3 | lr = 0.00250 | MAE tune = 1.76

,lam,epoca_migliore,mae_tune,distanza_dal_migliore
0,0.00000,7,1.76309,0.00021
1,0.00000,7,1.76302,0.00014
2,0.00001,5,1.76288,0.00000
3,0.00010,5,1.76566,0.00278
4,0.00100,5,1.78218,0.01929


lambda scelto: 1e-05


## 6. Modello finale: riaddestramento su train + tune

Con le impostazioni scelte sul tune (λ = `LAM_SCELTO`, passo dimezzato a ogni epoca, numero di epoche = epoca migliore per quel λ), riaddestro **una sola volta** su `train_enc` + `tune_enc` (~4 M flow). Così il modello impara anche dalle 12 ore subito prima della validation.

Senza early stopping: il tune ora è tra i dati di addestramento e non può più fare da giudice. Come controllo stampo la MAE media sui mini-batch di ogni epoca. $\mu$ e $\sigma$ di `start_rtt` restano quelli di `train_enc`.

*Nota:* anche così B2 impara da un campione di 4 M flow, mentre B1 usava tutto il train (46,7 M). Differenza da dichiarare.

In [9]:
EPOCHE = int(tab_lam.loc[tab_lam["lam"] == LAM_SCELTO, "epoca_migliore"].iloc[0])
print(f"impostazioni finali: lambda = {LAM_SCELTO:g}, epoche = {EPOCHE}, gamma = 0.5")

# train + tune, uno sotto l'altro
Xt_tt = torch.cat([Xt_tr, torch.from_numpy(X_tu).long()])
xt_tt = torch.cat([xt_tr, torch.from_numpy(x_tu)])
yt_tt = torch.cat([yt_tr, torch.from_numpy(y_tu)])
print("flow di addestramento:", len(yt_tt))

def addestra_finale(lam, n_epoche, lr=0.01, gamma=0.5, batch=4096, seme=0):
    torch.manual_seed(seme)
    modello = Lineare(N_CODICI, float(yt_tt.median()))
    opt = torch.optim.Adam([
        {"params": modello.emb.parameters(),   "weight_decay": lam},
        {"params": [modello.w, modello.b],     "weight_decay": 0.0},
    ], lr=lr)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(opt, gamma=gamma)

    n = len(yt_tt)
    for epoca in range(1, n_epoche + 1):
        t0 = time.time()
        modello.train()
        ordine = torch.randperm(n)
        somma_errori = 0.0
        for i in range(0, n, batch):
            idx  = ordine[i:i + batch]
            pred = modello(Xt_tt[idx], xt_tt[idx])
            loss = (pred - yt_tt[idx]).abs().mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
            somma_errori += loss.item() * len(idx)

        print(f"epoca {epoca} | lr = {scheduler.get_last_lr()[0]:.5f} | "
              f"MAE media sui batch = {somma_errori / n:.4f} | {time.time() - t0:.0f} s")
        scheduler.step()
    return modello

t0 = time.time()
modello_finale = addestra_finale(LAM_SCELTO, EPOCHE)
TEMPO_TRAIN_B2 = time.time() - t0
print(f"tempo di addestramento: {TEMPO_TRAIN_B2:.0f} s")

impostazioni finali: lambda = 1e-05, epoche = 5, gamma = 0.5
flow di addestramento: 4000667
epoca 1 | lr = 0.01000 | MAE media sui batch = 1.9964 | 2 s
epoca 2 | lr = 0.00500 | MAE media sui batch = 1.9259 | 2 s
epoca 3 | lr = 0.00250 | MAE media sui batch = 1.9247 | 2 s
epoca 4 | lr = 0.00125 | MAE media sui batch = 1.9242 | 2 s
epoca 5 | lr = 0.00063 | MAE media sui batch = 1.9240 | 2 s
tempo di addestramento: 11 s


### 6.1 Controllo: difficoltà del train e del tune

La MAE media sui batch del riaddestramento (1,924) è più **alta** della MAE sul tune trovata prima (1,763). Verifico se dipende dal periodo: calcolo, per il modello finale e per B0, la MAE separatamente sui flow di `train_enc` e di `tune_enc`. Se anche B0 sbaglia molto di più sul train, la differenza è dovuta ai dati (periodi di difficoltà diversa), non al modello.

In [10]:
mediana_tt = float(yt_tt.median())
for nome, X, x, y in [("train_enc", X_tr, x_tr, y_tr), ("tune_enc", X_tu, x_tu, y_tu)]:
    mae_b2 = np.abs(prevedi(modello_finale, X, x) - y).mean()
    mae_b0 = np.abs(mediana_tt - y).mean()
    print(f"{nome:10s} | B0 = {mae_b0:.4f} | B2 finale = {mae_b2:.4f} | riduzione = {1 - mae_b2 / mae_b0:.1%}")

train_enc  | B0 = 3.3648 | B2 finale = 1.9858 | riduzione = 41.0%
tune_enc   | B0 = 3.2090 | B2 finale = 1.7382 | riduzione = 45.8%


# 7. Valutazione sulla validation
### 7.1 Previsioni

Il modello finale calcola una previsione per ciascuno dei 10.069.254 flow della validation (a blocchi, senza gradienti: il modello non impara nulla da questi dati). Controllo che ci sia una previsione per ogni flow, che siano tutte numeri finiti e che la loro distribuzione sia ragionevole rispetto a quella di delta_rtt.

In [11]:
t0 = time.time()
pred_va = prevedi(modello_finale, X_va, x_va)
print(f"previsioni calcolate: {len(pred_va):,} in {time.time() - t0:.0f} s")

assert len(pred_va) == len(riga_va), "numero di previsioni diverso dal numero di flow"
assert np.isfinite(pred_va).all(), "ci sono previsioni NaN o infinite"

riepilogo = pd.DataFrame({
    "previsioni":     pd.Series(pred_va).describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]),
    "delta_rtt vero": pd.Series(y_va).describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]),
})
display(riepilogo.round(3))

previsioni calcolate: 10,069,254 in 1 s


,previsioni,delta_rtt vero
count,1.006925e+07,1.006925e+07
mean,3.256000e+00,3.380000e+00
std,2.318000e+00,4.727800e+01
min,-4.988000e+00,-2.108184e+03
1%,-1.420000e-01,-4.537000e+00
25%,2.270000e-01,6.400000e-02
50%,4.634000e+00,4.442000e+00
75%,5.207000e+00,5.256000e+00
99%,5.494000e+00,2.421800e+01
max,9.917000e+00,3.251811e+04


### 7.2 Previsioni accanto ai dati della validation

Metto le previsioni in una tabella (riga, pred) e creo in DuckDB la vista pred_b2, che le unisce a val_enc tramite riga (la chiave salvata nel file, che identifica sempre lo stesso flow). La vista contiene, per ogni flow, valore vero, previsione e le colonne dei sottogruppi: è ciò che serve a valuta. Il file non viene modificato. Controllo che tutti i 10.069.254 flow abbiano esattamente una previsione.

In [12]:
pred_df = pd.DataFrame({"riga": riga_va, "pred": pred_va})
con.register("pred_df", pred_df)

con.execute("""
    CREATE OR REPLACE VIEW pred_b2 AS
    SELECT v.riga, v.delta_rtt, p.pred,
           v.server_site, v.client_site, v.verso_incerto, v.an_durata_s
    FROM val_enc AS v
    LEFT JOIN pred_df AS p ON v.riga = p.riga
""")

controllo = con.execute("""
    SELECT COUNT(*)                            AS flow,
           COUNT(pred)                         AS con_previsione,
           COUNT(*) - COUNT(DISTINCT riga)     AS righe_duplicate
    FROM pred_b2
""").df()
display(controllo)

display(con.execute("SELECT * FROM pred_b2 LIMIT 5").df())

,flow,con_previsione,righe_duplicate
0,10069254,10069254,0


,riga,delta_rtt,pred,server_site,client_site,verso_incerto,an_durata_s
0,1,5.782,5.039165,CERN-PROD,MISSING,False,578.426916
1,2,4.399,3.755176,Nebraska,CERN-PROD,False,38.695331
2,3,4.904,5.264157,CERN-PROD,FZK-LCG2,False,430.632376
3,4,5.240,5.612564,CERN-PROD,MISSING,False,301.692160
4,5,4.962,4.406106,GLOW,MISSING,False,2.883743


### 7.3 Metriche sulla validation

Applico a pred_b2 la funzione valuta del notebook delle baseline, copiata senza modifiche: stesse metriche (MAE, RMSE, RMSE robusto †, P90, P99) e stessi sottogruppi (globale, FNAL, sito mancante, verso incerto, flow > 1 min). Le soglie LO e HI dell'RMSE robusto sono ricalcolate sul train completo da flows_prepared.parquet (quantili 0,05 % e 99,95 %), come per B0 e B1. Confronto della MAE globale con B0 (3,298) e B1 (2,209).

In [13]:
# --- Soglie dell'RMSE robusto (train completo, come nel notebook delle baseline) ---
PREPARED = PROJECT_ROOT / "data" / "student_dataset" / "flows_prepared.parquet"
Q_LO, Q_HI = 0.0005, 0.9995
LO, HI = con.execute(f"""
    SELECT quantile_cont(delta_rtt, [{Q_LO}, {Q_HI}])
    FROM read_parquet('{PREPARED.as_posix()}')
    WHERE split = 'train'
""").fetchone()[0]
print(f"LO = {LO:.2f} | HI = {HI:.2f}   (attesi: -184.23 | 58.82)")

NOTA_ROBUSTO = (f"† RMSE robusto: solo flow con delta_rtt in [{LO:.2f}, {HI:.2f}] ms, "
                f"quantili {Q_LO:.2%}–{Q_HI:.2%} del train (insieme fisso, uguale per tutti i modelli).")

# --- Funzione di valutazione (identica al notebook delle baseline) ---
SOTTOGRUPPI = {
    "globale":       "TRUE",
    "FNAL":          "server_site = 'USCMS-FNAL-WC1' OR client_site = 'USCMS-FNAL-WC1'",
    "sito mancante": "server_site = 'MISSING' OR client_site = 'MISSING'",
    "verso incerto": "verso_incerto",
    "flow > 1 min":  "an_durata_s > 60",
}

def valuta(vista):
    """vista: tabella/vista DuckDB con delta_rtt, pred e le colonne dei sottogruppi (solo validation)."""
    parti = []
    for ordine, (nome, cond) in enumerate(SOTTOGRUPPI.items()):
        parti.append(f"""
            SELECT {ordine} AS ordine, '{nome}' AS gruppo, COUNT(*) AS n,
                   AVG(ABS(delta_rtt - pred))                            AS mae,
                   SQRT(AVG(POWER(delta_rtt - pred, 2)))                 AS rmse,
                   SQRT(AVG(POWER(delta_rtt - pred, 2))
                        FILTER (WHERE delta_rtt BETWEEN {LO} AND {HI})) AS rmse_robusto,
                   quantile_cont(ABS(delta_rtt - pred), 0.90)            AS p90_abs,
                   quantile_cont(ABS(delta_rtt - pred), 0.99)            AS p99_abs
            FROM {vista}
            WHERE {cond}
        """)
    query = "SELECT * FROM (" + " UNION ALL ".join(parti) + ") ORDER BY ordine"
    return con.execute(query).df().drop(columns="ordine")

NOMI_METRICHE = {"mae": "MAE", "rmse": "RMSE", "rmse_robusto": "RMSE robusto †",
                 "p90_abs": "P90 |err|", "p99_abs": "P99 |err|"}

# --- Valutazione di B2 ---
ris_b2 = valuta("pred_b2")
display(ris_b2.rename(columns=NOMI_METRICHE).round(3))
print(NOTA_ROBUSTO)

mae_b2 = ris_b2.loc[ris_b2["gruppo"] == "globale", "mae"].iloc[0]
print(f"\nMAE sulla validation → B0: 3.298 | B1: 2.209 | B2: {mae_b2:.3f}")

LO = -184.23 | HI = 58.82   (attesi: -184.23 | 58.82)


,gruppo,n,MAE,RMSE,RMSE robusto †,P90 |err|,P99 |err|
0,globale,10069254,1.966,47.222,5.264,4.599,23.034
1,FNAL,215471,1.268,2.803,2.202,2.960,6.452
2,sito mancante,2122477,2.210,16.351,6.360,4.818,25.333
3,verso incerto,32662,0.397,7.530,1.923,0.467,2.431
4,flow > 1 min,6632723,2.539,57.992,6.222,4.963,26.748


† RMSE robusto: solo flow con delta_rtt in [-184.23, 58.82] ms, quantili 0.05%–99.95% del train (insieme fisso, uguale per tutti i modelli).

MAE sulla validation → B0: 3.298 | B1: 2.209 | B2: 1.966


In [14]:
OUT_DIR = PROJECT_ROOT / "results" / "metriche"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# metriche per sottogruppo
ris_b2.to_csv(OUT_DIR / "B2_lineare_sottogruppi.csv", index=False)

# riepilogo: riga globale + impostazioni
glob = ris_b2.set_index("gruppo").loc["globale"]
riepilogo_b2 = {
    "modello":           "B2 lineare (L1)",
    "MAE":               float(glob["mae"]),
    "RMSE":              float(glob["rmse"]),
    "RMSE_robusto":      float(glob["rmse_robusto"]),
    "P90_abs":           float(glob["p90_abs"]),
    "P99_abs":           float(glob["p99_abs"]),
    "parametri":         int(sum(p.numel() for p in modello_finale.parameters())),
    "tempo_train_s":     round(TEMPO_TRAIN_B2, 1),
    "flow_addestramento": int(len(yt_tt)),
    "lambda":            LAM_SCELTO,
    "epoche":            EPOCHE,
    "gamma":             0.5,
    "soglie_rmse_robusto": [float(LO), float(HI)],
}
with open(OUT_DIR / "B2_lineare.json", "w", encoding="utf-8") as f:
    json.dump(riepilogo_b2, f, ensure_ascii=False, indent=2)

print("salvati in", OUT_DIR)
print(json.dumps(riepilogo_b2, indent=2))

salvati in /Users/giovannipavan/Desktop/EPFL/Corsi/Primo Semestre/ML/projects/flow-tokenization/results/metriche
{
  "modello": "B2 lineare (L1)",
  "MAE": 1.966399624044382,
  "RMSE": 47.22161084222985,
  "RMSE_robusto": 5.263721887524995,
  "P90_abs": 4.598690421390536,
  "P99_abs": 23.0344129575348,
  "parametri": 1733,
  "tempo_train_s": 10.5,
  "flow_addestramento": 4000667,
  "lambda": 1e-05,
  "epoche": 5,
  "gamma": 0.5,
  "soglie_rmse_robusto": [
    -184.23263,
    58.82300000000002
  ]
}
